# Check Accuracy (simple algorithm)

Run this notebook in the same environment as notebook 2. Run it once with the base model and once with your finetuned model.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Schnowflake/playing_cards_excercise", split="test")

In [ ]:
print(dataset[0])

In [ ]:
print(len(dataset))

# Load Model

In [ ]:
# Which model do you want to evaluate?
#   "unsloth/Qwen3.5-0.8B"  -> the base model (not finetuned)
# model_name="lora_model_cards"    #-> your finetuned model from notebook 2
model_name = "unsloth/Qwen3.5-0.8B"

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name,
    load_in_4bit = False, # Unsloth recommends 16bit LoRA for Qwen3.5 (no 4bit)
)
FastVisionModel.for_inference(model)  # Enable for inference!

In [ ]:
def VLM_Call(image):

    instruction = """What cards are in the image? 
    Answer in the same format as these three examples:
    2 of diamond, 4 of club, 6 of club
    King of hearts, 5 of spades, 9 of diamond
    9 of club, Ace of Spades, 2 of hearts

    What you can see in the image:
    """

    messages = [
        {
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": instruction}],
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    result = model.generate(**inputs, max_new_tokens = 256, use_cache=True,
                            do_sample=False)  # greedy decoding -> same answer every run

    # Only decode the new tokens (not our prompt)
    new_tokens = result[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Qwen3.5 can "think" before answering - only keep the part after </think>
    answer = answer.split("</think>")[-1].strip()

    return answer

In [ ]:
image = dataset[1]["image"]

print(dataset[1]["qa"][0]["answer"])
display(image)
print(VLM_Call(image))

In [ ]:
import re

def correct_or_not(groundtruth, vlm_answer):
    # Normalize both strings: lowercase
    gt = groundtruth.lower().strip()
    answer = vlm_answer.lower().strip()

    # Extract all cards from the groundtruth (pattern: e.g. "a of club", "10 of hearts", "q of spade")
    cards = re.findall(r'\b(?:[2-9]|10|[jqka]) of [a-z]+\b', gt)

    # Check that every card from groundtruth appears in the answer
    if all(card in answer for card in cards):
        return "correct!"
    else:
        return "wrong!"

In [ ]:
total_correct = 0

data_to_safe = []

for row in dataset:
    vlm_answer = VLM_Call(row["image"])
    judge_answer = correct_or_not(row["qa"][0]["answer"], vlm_answer)
    if judge_answer == "correct!":
        total_correct += 1
        print("correct!")
    else:
        print("Not correct!")

    data = {}
    data["Groundtruth"] = row["qa"][0]["answer"]
    data["VLM"] = vlm_answer
    data["Judge"] = judge_answer

    data_to_safe.append(data)

print(f"{total_correct}/{len(dataset)} correct")

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(data_to_safe)
df.to_csv(f"Results_{model_name.split('/')[-1]}_easyALG.csv", sep=',', encoding='utf-8', index=False, header=True)